# SONGS — Initialisation notebook

This short notebook demonstrates how to initialise the `SONGS` generator, create a small set of synthetic IFU spectral cubes, and visualise one of them. The cells are documented with comments and explanatory markdown to make it easy to adapt the workflow for pretraining or quick experiments.

Requirements:
- `SONGS` installed in the active Python environment (or `pip install -e .` from the repo root)
- Optional plotting packages: `matplotlib`, `astrodendro` for diagnostics

What the notebook does:

1. Initialise a `SONGS` instance with a chosen configuration.
2. Generate a small number of cubes using `generate_cubes()`.
3. Visualise a selected cube with the bundled `visualise` helper.

By default each cube now includes **diffuse emission**: a faint stellar/gaseous halo around the central galaxy, diffuse bridges connecting the central galaxy to each satellite (thick at the halo end, narrow at the satellite end), and tidal-tail lookalikes extending from each satellite away from the central. The default amplitudes keep the diffuse features well below the satellites in surface brightness. Pass `diffuse_params={'enabled': False}` to disable, or override individual knobs (see `songs.core.DEFAULT_DIFFUSE_PARAMS`).

Read the inline comments for parameter meanings and quick tips on saving/adjusting visuals.

In [1]:
import songs  # package name (ensure the package is importable in this environment)
import numpy as np

# Initialise the generator using the convenience wrapper in __init__.py.
# Parameters explained:
# - n_gals: number of galaxy components per cube (None -> random 1-3)
# - n_cubes: how many cubes to generate in this run
# - resolution: 'all'|'resolved'|'unresolved' (controls Re/beam ratio sampling)
# - grid_size: spatial size of output cube (pixels)
# - diffuse_params: optional dict overriding defaults for halo/bridges/tails.
#       Pass {'enabled': False} to disable diffuse emission entirely, or override
#       individual knobs (e.g. 'halo_Se_factor', 'bridge_width_start_factor',
#       'tail_curvature'). Defaults live in songs.core.DEFAULT_DIFFUSE_PARAMS.

seed = np.random.randint(0,1000)
print(seed)

g = songs.init(
    n_gals=3,
    n_cubes=1,
    resolution='resolved',
    grid_size=96,
    n_spectral_slices=64,
    offset_gals=75,
    seed=741,#seed,
    save=True,  # optional fname='...' to specify location
    # diffuse_params={'enabled': False},   # uncomment to turn off diffuse emission
    # diffuse_params={                     # or customise individual knobs
    #     'halo_Se_factor': 0.04,
    #     'bridge_width_start_factor': 3.5,
    #     'bridge_width_end_factor': 0.8,
    #     'tail_curvature': 0.5,
    # },
)

# Generate the cubes. This runs the pipeline and returns a list of tuples (cube, metadata).
# Each `cube` has shape (n_velocity, ny, nx). `metadata` carries keys like 'average_vels',
# 'beam_info', 'pix_spatial_scale', 'n_gals', and 'diffuse_emission' (bool indicating
# whether the halo/bridges/tails were added).
sim = g.generate_cubes()

print('Number of cubes generated:', len(sim))
print('First cube shape:', sim[0][0].shape)
print('Diffuse emission present:', sim[0][1].get('diffuse_emission'))


import songs.visualise as visualise

# Now you can assign the functions directly
visualise.slice_view(sim, idx=0)


164

          _____       _    _____      _             _____            __ _   
         / ____|     | |  / ____|    | |           / ____|          / _| |  
        | |  __  __ _| | | |    _   _| |__   ___  | |     _ __ __ _| |_| |_ 
        | | |_ |/ _` | | | |   | | | | '_ \ / _ \ | |    | '__/ _` |  _| __|
        | |__| | (_| | | | |___| |_| | |_) |  __/ | |____| | | (_| | | | |_ 
         \_____|\__,_|_|  \_____\__,_|_.__/ \___|  \_____|_|  \__,_|_|  \__|
        


§------------ Creating cube # 1 ------------§
Creating disk #1...
Calculating the flux density values at each spatial location
Calculating and assigning velocity vectors...
Rotating 51.33 degrees about X axis and 7.00 degrees about Y axis:
1. Rotating/transforming the whole system...
2: Rotating the individual velocity vectors...
Disk #1 generated!

Creating disk #2...
Calculating the flux density values at each spatial location
Calculating and assigning velocity vectors...
Rotating 67.84 degrees about X axis and -97

(<Figure size 600x600 with 2 Axes>,
 <Axes: title={'center': '$\\rm Channel\\ 32\\;:\\;v=-9.4\\;km\\;s^{-1}$'}>)

# Explanation of the generation step

The `songs.init(...)` call above creates a `SONGS` object with the specified configuration. Key points:

- `n_gals=3` requests three galaxy components per cube (a central galaxy plus two satellites).
- `n_cubes=1` instructs the pipeline to create one cube in this run.
- `resolution='resolved'` biases the sampling so the galaxies are larger relative to the beam (use `'unresolved'` or `'all'` for other regimes).
- `grid_size` controls the output spatial grid in pixels; smaller values reduce memory and speed up generation.
- `seed` is provided to make results reproducible.
- `diffuse_params` (optional) controls the halo / bridges / tidal-tail components. By default they are enabled with conservative amplitudes (see `songs.core.DEFAULT_DIFFUSE_PARAMS`). Bridges emerge from the halo (thick end) and narrow toward the satellite without extending past it.

After calling `generate_cubes()`, `sim` is a list where each element is `(cube_array, metadata_dict)`. Use `cube_array.shape` to inspect dimensions and `metadata_dict['average_vels']` to get the channel velocity centres. `metadata_dict['diffuse_emission']` reports whether diffuse emission was added.


# Next: visualise the generated cube.


In [2]:
import songs.visualise as visualise

# Now you can assign the functions directly
visualise.moment0(sim, idx=0)

visualise.moment1(sim, idx=0)

visualise.spectrum(sim, idx=0)

(<Figure size 700x450 with 1 Axes>,
 <Axes: xlabel='(Line-of-sight) Velocity ($\\rm km\\;s^{-1}   $)', ylabel='Flux Density ($\\rm Jy\\;beam^{-1}$)'>)

## Visualization notes

The `visualise` helper computes a dendrogram-based mask to isolate emission and then shows:

- Moment-0 (integrated flux) with a scalebar and beam ellipse
- Moment-1 (intensity-weighted velocity) map
- Integrated velocity spectrum (flux vs velocity)

If you run headless or on a remote server, call `g.visualise(..., save=True, fname_save='my_figures')` to save PDFs instead of showing interactive windows.

You can also inspect `sim` directly to make custom plots with `matplotlib` using the arrays and metadata.